# 05 · The methods: tree, ladder, bridge, chance reference, fair comparison

**Produces:** the numbers behind your claim, each with its denominator and period, and the chart for slide 4.

**Needs:** notebook 03 run.

Five methods from Weeks 1 and 2, written vectorised so they run on the whole pack in seconds. Use the one your brief needs; read the others so you can follow the other groups on presentation day.

## CONFIG

In [1]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "A",
    "group": "4",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

bk = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at", "collected_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
bk["half"] = np.where(bk["slot_at"].between(*CONFIG["period_last"]), "H1_2025", np.where(bk["slot_at"].between(*CONFIG["period_this"]), "H1_2026", "other"))
bk = bk[bk["half"] != "other"]
print(bk.groupby("half").agg(bookings=("booking_id", "size"), tests=("n_tests", "sum"), people=("person_id", "nunique")))

3.0.6 2.5.4
         bookings   tests  people
half                             
H1_2025    187771  431261   93926
H1_2026    193005  452933   97501


## 1 · The tree (brief A)

Tests = people × visits per person × tests per visit. Growth in a product is close to the sum of growths in the factors (exactly so in logs). Compute each factor for both halves and check that they multiply back.

In [2]:
def tree(df, by=None):
    g = df.groupby(([by] if by else []) + ["half"], observed=True).agg(tests=("n_tests", "sum"), visits=("booking_id", "size"), people=("person_id", "nunique"))
    g["visits_per_person"] = g["visits"] / g["people"]
    g["tests_per_visit"] = g["tests"] / g["visits"]
    return g

t = tree(bk)
growth = (t.xs("H1_2026") / t.xs("H1_2025") - 1) * 100
print(growth.round(2))
# log-decomposition: the branch growths sum to the total growth exactly
logs_ = np.log(t.xs("H1_2026") / t.xs("H1_2025"))
checks.check("branches multiply back to the total (log decomposition)", abs(logs_["people"] + logs_["visits_per_person"] + logs_["tests_per_visit"] - logs_["tests"]) < 1e-9,
             f"people {logs_['people']:+.4f} + visits/person {logs_['visits_per_person']:+.4f} + tests/visit {logs_['tests_per_visit']:+.4f} = {logs_['tests']:+.4f}")

tests                5.03
visits               2.79
people               3.81
visits_per_person   -0.98
tests_per_visit      2.18
dtype: float64
PASS  branches multiply back to the total (log decomposition)   people +0.0374 + visits/person -0.0099 + tests/visit +0.0215 = +0.0490


True

In [3]:
by_city = tree(bk, "city")
city_growth = ((by_city.xs("H1_2026", level="half") / by_city.xs("H1_2025", level="half") - 1) * 100).round(1)
city_growth[["tests", "people", "visits_per_person", "tests_per_visit"]].sort_values("tests")

,tests,people,visits_per_person,tests_per_visit
city,,,,
Jaipur,-10.6,-12.0,-0.1,1.8
Chennai,0.9,-4.8,3.1,2.8
Hyderabad,7.2,8.1,-2.5,1.8
Bengaluru,7.6,7.2,-2.3,2.8
Pune,8.3,8.9,-2.2,1.6
Ahmedabad,10.5,9.7,-1.1,1.8


> **Interview angle.** Revenue grew 10 percent and patients grew 4. Decompose the rest without a calculator, then say what you would check before presenting it.
>
> *What a good answer covers:* Roughly 6 points sit in visits per patient and revenue per visit together; log growth makes the split exact. Check that the patient count is a count of people and not of ids (duplicates), that both periods use the same definition of a visit, and that a system change did not move the grain.

## 2 · The ladder (brief B)

Confirm the drop, compare like with like, decompose, isolate, hypothesise. In code the ladder is a sequence of groupbys where each step changes one thing: the grain, the window, the segment.

In [4]:
bk["quarter"] = bk["slot_at"].dt.to_period("Q").astype(str)
q = bk[bk["city"].isin(["Chennai", "Jaipur"])].groupby(["city", "quarter", "system"], observed=True).agg(bookings=("booking_id", "size"), tests=("n_tests", "sum")).reset_index()
pivot = q.pivot_table(index=["city", "quarter"], columns="system", values=["bookings", "tests"], aggfunc="sum", fill_value=0)
pivot

bookings         tests       
system            legacy   nova legacy   nova
city    quarter                              
Chennai 2025Q1     17363      0  39950      0
        2025Q2     17221      0  39393      0
        2026Q1     17545      0  41476      0
        2026Q2         0  16408      0  38579
Jaipur  2025Q1     10387      0  23869      0
        2025Q2      9025      0  20779      0
        2026Q1      8750      0  20369      0
        2026Q2         0   8309      0  19556

In [5]:
# Rung 2, like with like: the same quarter a year earlier, per city, at booking grain, after the identity rule
yoy = bk[bk["city"].isin(["Chennai", "Jaipur"])].groupby(["city", "quarter"], observed=True)["booking_id"].size().unstack("quarter")
def pct_change(table, later, earlier, label):
    """Like with like means both periods must exist; say so when one does not, rather than fail quietly."""
    if later in table and earlier in table:
        print(((table[later] / table[earlier] - 1) * 100).round(1).rename(label))
    else:
        print(f"{label}: period {earlier if earlier not in table else later} is not in this pack (sample data?); nothing to compare yet")
pct_change(yoy, "2026Q2", "2025Q2", "Q2 2026 vs Q2 2025, bookings, %")

city
Chennai   -4.7
Jaipur    -7.9
Name: Q2 2026 vs Q2 2025, bookings, %, dtype: float64


## 3 · The bridge (brief C)

From invoiced to collected, every bar a count and an amount, and the two sides reconcile.

In [6]:
inv = pd.read_csv(CONFIG["clean"] / "invoices_clean.csv.gz", parse_dates=["invoice_date"], dtype={"invoice_id": "string"})
pay = pd.read_csv(CONFIG["clean"] / "payments_clean.csv.gz", parse_dates=["created_at"], dtype={"invoice_id": "string"})
q2 = inv[inv["invoice_date"].between("2026-04-01", "2026-06-30")]
cap = pay[pay["status"] == "captured"]
cutoff = pd.Timestamp("2026-06-30 23:59:59")
m = q2.merge(cap.groupby("invoice_id").agg(paid_rs=("amount_rs", "sum"), n_pay=("payment_id", "size"), first_paid=("created_at", "min")), on="invoice_id", how="left")
unpaid = m["paid_rs"].isna()
paid_after_cutoff = m["first_paid"] > cutoff
steps = {
    "invoices never paid": -m.loc[unpaid, "amount_rs"].sum(),
    "paid after the quarter closed (timing)": -m.loc[~unpaid & paid_after_cutoff, "amount_rs"].sum(),
    "over- or under-paid (amount differences)": (m.loc[~unpaid & ~paid_after_cutoff, "paid_rs"] - m.loc[~unpaid & ~paid_after_cutoff, "amount_rs"]).sum(),
}
collected_in_q = m.loc[~unpaid & ~paid_after_cutoff, "paid_rs"].sum()
checks.bridge_balances("Q2 2026 invoiced to collected", q2["amount_rs"].sum(), steps, collected_in_q)
orphans = cap[~cap["invoice_id"].isin(inv["invoice_id"])]
print(f"captured payments matching no invoice: {len(orphans):,} rows, Rs {orphans['amount_rs'].sum():,.0f}")

      invoices never paid                        -11,107,904.00
      paid after the quarter closed (timing)      -2,982,123.00
      over- or under-paid (amount differences)       529,126.00
PASS  Q2 2026 invoiced to collected: bridge balances   start 132,847,274.00 + steps -13,560,901.00 = 119,286,373.00 against end 119,286,373.00
captured payments matching no invoice: 4,588 rows, Rs 7,269,408


> **Interview angle.** Finance says collected is 4 percent below invoiced. Before you call it a leak, what three things do you separate it into?
>
> *What a good answer covers:* Timing (invoiced in the period, paid after the cutoff), unmatched (payments whose reference does not resolve to an invoice, and invoices with no payment at all), and amount differences (fees, partial payments, retries). Each is a bar with a count and an amount, and only the middle one is a possible leak.

## 4 · The chance reference (brief D)

How often would a clinic of this size look this bad by chance alone? Shuffle the clinic labels across bookings many times with NumPy, recompute the clinic's rate each time, and see where the real rate falls. Vectorised: one 2-D array of permutations, no Python loop over shuffles.

In [7]:
d = bk[bk["channel"] == "clinic"].copy()
d["no_show"] = (d["status"] == "no_show").astype(int)
rates = d.groupby("clinic_id", observed=True)["no_show"].agg(["mean", "size"]).rename(columns={"mean": "rate", "size": "bookings"})
network = d["no_show"].mean()
rates["diff_pts"] = (rates["rate"] - network) * 100
print(f"network no-show rate {network:.3f}")
print(rates.sort_values("rate", ascending=False).head(5).round(3))

network no-show rate 0.164
            rate  bookings  diff_pts
clinic_id                           
CHE-03     0.190      7411     2.566
PUN-04     0.172      4573     0.790
HYD-03     0.171      7072     0.633
AHM-05     0.170      4779     0.592
AHM-02     0.169      9558     0.446


In [8]:
def shuffle_reference_loop(outcomes: np.ndarray, n_clinic: int, n_shuffles: int, rng) -> np.ndarray:
    """The literal shuffle: relabel n_clinic random bookings as 'this clinic', take their rate, repeat. Clear, and slow."""
    return np.array([outcomes[rng.choice(len(outcomes), n_clinic, replace=False)].mean() for _ in range(n_shuffles)])

def shuffle_reference(outcomes: np.ndarray, n_clinic: int, n_shuffles: int, rng) -> np.ndarray:
    """The same null, vectorised. For a 0/1 outcome, drawing n_clinic bookings without replacement from the network and
    counting no-shows is a hypergeometric draw, which NumPy produces in one call. Same distribution as the loop, a thousand times faster."""
    ngood = int(outcomes.sum()); nbad = int(len(outcomes) - ngood)
    return rng.hypergeometric(ngood, nbad, n_clinic, size=n_shuffles) / n_clinic

worst = rates["rate"].idxmax()
y = d["no_show"].to_numpy()
t0 = time.time(); ref_loop = shuffle_reference_loop(y, int(rates.loc[worst, "bookings"]), 200, rng); t_loop = time.time() - t0
t0 = time.time(); ref = shuffle_reference(y, int(rates.loc[worst, "bookings"]), 20000, rng); t_vec = time.time() - t0
print(f"loop: 200 shuffles in {t_loop:.2f}s, mean {ref_loop.mean():.4f}; vectorised: 20,000 draws in {t_vec:.3f}s, mean {ref.mean():.4f}")
real = rates.loc[worst, "rate"]
p = float((ref >= real).mean())
print(f"{worst}: real rate {real:.3f} on {int(rates.loc[worst, 'bookings']):,} bookings; shuffled rates mean {ref.mean():.3f}, 95% band {np.percentile(ref, 2.5):.3f} to {np.percentile(ref, 97.5):.3f}; share of shuffles at least as extreme: {p:.4f}")

loop: 200 shuffles in 0.05s, mean 0.1647; vectorised: 20,000 draws in 0.003s, mean 0.1642
CHE-03: real rate 0.190 on 7,411 bookings; shuffled rates mean 0.164, 95% band 0.156 to 0.173; share of shuffles at least as extreme: 0.0000


In [9]:
# Like with like: the same slot, the same source. A rate that survives this comparison is about the clinic; one that does not is about its mix.
d["slot_bucket"] = pd.cut(d["slot_at"].dt.hour, [0, 10, 14, 24], labels=["morning", "midday", "evening"])
mix_rate = d.groupby(["slot_bucket", "source"], observed=True)["no_show"].mean().rename("network_rate_for_mix").reset_index()
d2 = d.merge(mix_rate, on=["slot_bucket", "source"], how="left", validate="many_to_one")
adj = d2.groupby("clinic_id", observed=True).agg(rate=("no_show", "mean"), expected_from_mix=("network_rate_for_mix", "mean"), bookings=("no_show", "size"))
adj["excess_after_mix_pts"] = (adj["rate"] - adj["expected_from_mix"]) * 100
adj.sort_values("excess_after_mix_pts", ascending=False).head(5).round(3)

,rate,expected_from_mix,bookings,excess_after_mix_pts
clinic_id,,,,
PUN-04,0.172,0.163,4573,0.880
HYD-03,0.171,0.163,7072,0.727
AHM-05,0.170,0.164,4779,0.600
AHM-02,0.169,0.163,9558,0.593
BLR-02,0.167,0.163,11236,0.391


> **Interview angle.** Why is drawing without replacement the same as shuffling labels here, and when would you have to shuffle for real?
>
> *What a good answer covers:* With a 0/1 outcome and one group of fixed size, the shuffled rate depends only on how many of the group's bookings are no-shows, which is a hypergeometric count. When the statistic is not a count of a binary outcome in one group, or when there are several groups compared at once, the closed form no longer applies and you permute the labels and recompute the statistic.

> **Interview angle.** What does your shuffle test assume, and what does the p-value it produces mean and not mean?
>
> *What a good answer covers:* It assumes bookings are exchangeable across clinics under the null, which fails if clinics differ in mix; hence the like-with-like step. The p-value is the share of shuffled worlds at least as extreme as the real one; it is not the probability the clinic is fine, and a small p on a tiny difference is not a reason to act.

## 5 · The fair comparison (brief E)

Reproduce marketing's number, then look at who got the offer, then split. The grid: offer or not, before or after, by segment. A lift that holds in every segment is a lift; one that holds only in the total is a mix shift.

In [10]:
camp = pd.read_csv(CONFIG["raw"] / "campaigns" / "campaign_bookings.csv.gz", dtype="string")
camp["booking_id"] = camp["booking_id"].str.strip()
bkq = bk.merge(camp[["booking_id", "campaign_id"]], on="booking_id", how="left", validate="one_to_one")
bkq["offer"] = bkq["campaign_id"].notna()
all_b = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
all_b["quarter"] = all_b["slot_at"].dt.to_period("Q").astype(str)
qq = all_b.groupby("quarter")["booking_id"].size()
r = lambda a, b: f"{(qq[a]/qq[b]-1)*100:+.1f}%" if a in qq and b in qq else "period missing in this pack"
print("marketing's comparison, Q1 2026 vs Q4 2025:", r("2026Q1", "2025Q4"), " | the same comparison a year earlier is not in the pack; Q1 2025 vs Q2 2025:", r("2025Q1", "2025Q2"))
# who got the offer: first-seen date per person, so 'new' is defined before the offer and cannot be changed by it
first_seen = all_b.groupby("person_id")["slot_at"].min()
bkq["is_new_patient"] = bkq["person_id"].map(first_seen) >= pd.Timestamp("2026-01-01")
pd.crosstab(bkq.loc[bkq["half"] == "H1_2026", "offer"], bkq.loc[bkq["half"] == "H1_2026", "is_new_patient"], normalize="index").round(3)

marketing's comparison, Q1 2026 vs Q4 2025: +11.0%  | the same comparison a year earlier is not in the pack; Q1 2025 vs Q2 2025: +2.5%


is_new_patient,False,True
offer,,
False,0.543,0.457
True,0.674,0.326


In [11]:
seg = all_b.groupby(["quarter", "city"])["booking_id"].size().unstack("quarter")
pct_change(seg, "2026Q1", "2025Q4", "Q1 2026 vs Q4 2025 by city, %")
print("\nIf every city shows about the same lift with and without the offer, the lift is the season, not the offer. Split by channel next.")

city
Ahmedabad    15.0
Bengaluru    18.7
Chennai       1.6
Hyderabad    14.9
Jaipur       -5.1
Pune         14.3
Name: Q1 2026 vs Q4 2025 by city, %, dtype: float64

If every city shows about the same lift with and without the offer, the lift is the season, not the offer. Split by channel next.


> **Interview angle.** Marketing shows bookings up 9 percent in the campaign quarter. Give three reasons that is not proof the campaign worked, and the comparison that would be.
>
> *What a good answer covers:* Seasonality (the same quarter rises every year), selection (the offer reached people who were already going to book), and mix (new patients grew for other reasons). The comparison that would show cause is like-for-like patients who did not get the offer in the same period: a holdout, or at least a matched before-behaviour comparison.

## 6 · SUM: the chart for slide 4

One chart, the claim's number against its comparison, with the denominator in the axis label. Save it to `slides/`.

In [12]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.2))
g = city_growth["tests"].sort_values()
ax.barh(g.index, g.values, color=["#D63A6A" if v < 0 else "#4E38AB" for v in g.values])
ax.axvline(0, color="#1A1440", lw=1)
ax.set_xlabel("Tests, H1 2026 vs H1 2025, percent change (denominator: tests on collected and no-show bookings, H1 2025)")
ax.set_title("Where the growth went, by city")
for i, v in enumerate(g.values): ax.text(v + (0.3 if v >= 0 else -0.3), i, f"{v:+.1f}%", va="center", ha="left" if v >= 0 else "right", fontsize=9)
plt.tight_layout()
(ROOT / "slides").mkdir(exist_ok=True)
fig.savefig(ROOT / "slides" / "slide4_chart.png", dpi=200)
print("saved slides/slide4_chart.png")
checks.summary()

saved slides/slide4_chart.png
2 of 2 checks passed


,check,passed,detail
0,branches multiply back to the total (log decom...,True,people +0.0374 + visits/person -0.0099 + tests...
1,Q2 2026 invoiced to collected: bridge balances,True,"start 132,847,274.00 + steps -13,560,901.00 = ..."


## Before you move on

- Which number is on slide 3, and what are its denominator and period? Say them aloud.
- Which of the five methods did you use, and why not the nearest other one? That is an approach-log row.
- What would make you wrong? That is slide 5.